In [1]:
import re, os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import HumanMessage

from dotenv import load_dotenv
load_dotenv()
#os.environ['OPENAI_API_KEY'] = ""

True

In [2]:
import json

input_file = "../dataset/ellipsis_recovered_formatted/test.json"
output_file = "../dataset/naturalized/ellipsis_recovered_test.json"

with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    request_timeout=60,
    api_key=os.environ["OPENAI_API_KEY"]
)

system_prompt = """
당신은 한국어 대화의 '표준화 도우미'입니다.
입력으로 최근 대화 맥락(최대 5턴)과, 방금 발화된 '마지막 발화'가 주어지면,
해당 마지막 발화를 더 자연스럽고 맥락이 풍부한 규범적인 한국어 문장으로 표준화하세요.

표준화 지침:
- 의미는 보존하되, 과도한 구어체/중복 감탄/자모 반복(ㅋㅋ, ㅎㅎ, ㅠㅠ 등)은 절제(최대 1회)하거나 적절한 문어체로 바꿉니다.
- 괄호 [] 안의 보충설명은 문장에 자연스럽게 흡수하고, 대괄호 기호는 제거합니다.
- 맞춤법, 띄어쓰기, 어미(종결어미 포함)를 바로잡습니다.
- 슬랭/축약(예: '글치'→'그렇지', '오오'→'오', '유리서버'는 그대로, 맥락상 고유명 유지) 등은 맥락을 해치지 않는 선에서 자연스러운 표현으로 바꿉니다.
- 말투는 존댓말/반말을 맥락에 맞게 유지하되, 한 대화 내 일관성을 지향합니다.
- 결과는 한두 문장 이내로 간결하게 마무리하고 문장부호로 끝냅니다.

출력 형식:
- 표준화된 발화만 그대로 출력 (여는/닫는 따옴표나 설명 금지)
"""

In [4]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

# system_prompt는 이미 위에서 정의하신 것을 그대로 사용
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt),
    HumanMessagePromptTemplate.from_template(
        """[대화 맥락(최대 4턴)]
{context_block}

[표준화 대상 마지막 발화]
{last_utterance}

지침에 따라 '표준화된 마지막 발화'만 출력하세요."""
    ),
])

chain = prompt | llm | StrOutputParser()


In [5]:
import json, re
from typing import List, Tuple, Dict
from tqdm import tqdm

def parse_dialogue_to_turns(dialogue: str) -> List[Tuple[str, str]]:
    """
    '화자1: ...' / '화자2: ...' 라인을 (speaker, text) 튜플 리스트로 파싱.
    레이블 없는 줄은 직전 화자 발화에 이어붙임.
    """
    turns = []
    for line in dialogue.split("\n"):
        line = line.strip()
        if not line:
            continue
        m = re.match(r"^(화자[12])\s*:\s*(.+)$", line)
        if m:
            spk, txt = m.group(1), m.group(2).strip()
            turns.append((spk, txt))
        else:
            if turns:
                turns[-1] = (turns[-1][0], f"{turns[-1][1]} {line}".strip())
    return turns

def soft_preclean(text: str) -> str:
    text = text.replace("\r", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_context_prev4(turns: List[Tuple[str, str]], idx: int) -> List[str]:
    """
    인덱스 idx의 발화를 표준화할 때, 직전 최대 4개 발화를 컨텍스트로 사용.
    반환: ["화자1: ...", "화자2: ...", ...]
    """
    start = max(0, idx - 4)
    return [f"{spk}: {txt}" for spk, txt in turns[start:idx]]

# records: 원본 JSON 레코드들 (이미 data 변수에 로드되어 있음)
utterances: List[Dict] = []   # 발화 단위 리스트

for rec_idx, rec in enumerate(data):
    rec_id = rec.get("id", f"rec_{rec_idx}")
    turns = parse_dialogue_to_turns(rec.get("dialogue", ""))

    for turn_idx, (spk, txt) in enumerate(turns):
        utterances.append({
            "record_index": rec_idx,      # 원본 레코드 인덱스
            "record_id": rec_id,          # 원본 레코드 ID
            "turn_index": turn_idx,       # 대화 내 발화 인덱스
            "speaker": spk,               # 화자1/화자2
            "text_raw": soft_preclean(txt)
        })

print(f"총 발화 개수: {len(utterances)}")


총 발화 개수: 3021


In [6]:
# 표준화 결과를 utterances 리스트에 병합 (키: standardized_text)
for rec_idx, rec in enumerate(tqdm(data, desc="Standardizing utterances (per turn)")):
    turns = parse_dialogue_to_turns(rec.get("dialogue", ""))

    for turn_idx, (spk, txt) in enumerate(turns):
        # utterances 리스트에서 해당 항목 찾아오기 (인덱스 매칭 방식)
        # 큰 데이터에서도 O(1)에 가깝게 하려면 별도 인덱스 맵을 만들 수 있으나,
        # 여기서는 간단히 조건으로 조회
        # (필요시 최초 생성 시 key=(rec_idx, turn_idx)로 dict 인덱스 구성 추천)
        for u in utterances:
            if u["record_index"] == rec_idx and u["turn_index"] == turn_idx:
                target_u = u
                break

        context_prev4 = build_context_prev4(turns, turn_idx)
        context_block = "\n".join(context_prev4) if context_prev4 else "(이전에 주어진 대화가 없습니다)"
        last_utt_for_prompt = f"{spk}: {soft_preclean(txt)}"

        std_txt = chain.invoke({
            "context_block": context_block,
            "last_utterance": last_utt_for_prompt
        }).strip()

        target_u["standardized_text"] = std_txt


Standardizing utterances (per turn):  47%|████▋     | 190/408 [1:00:09<1:09:00, 18.99s/it]


KeyboardInterrupt: 

In [ ]:
# 레코드별로 표준화된 대화 재구성
standardized_dialogues = {}  # rec_idx -> "화자1: ...\n화자2: ...\n..."
for u in utterances:
    rec_idx = u["record_index"]
    line = f"{u['speaker']}: {u.get('standardized_text', u['text_raw'])}"
    standardized_dialogues.setdefault(rec_idx, []).append(line)

# 원본 data에 'dialogue' 필드 수정
for rec_idx, rec in enumerate(data):
    std_lines = standardized_dialogues.get(rec_idx, [])
    rec["dialogue"] = "\n".join(std_lines)

# 덮어쓰기가 걱정되면 다른 파일 경로로 저장하세요.
# 여기서는 사용자가 지정한 output_file을 그대로 사용합니다.
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {output_file}")

저장 완료: ../dataset/naturalized/ellipsis_recovered_train.json
